# Redis Caching Patterns

## Overview
Caching is the most common Redis use case — storing frequently-read, slow-to-compute data in Redis to reduce database load and latency.

**Caching strategies:**

| Pattern | Description | Use when |
|---|---|---|
| **Cache-aside** (lazy) | App checks cache; on miss fetches DB and stores | Most common; simple; handles cold starts |
| **Write-through** | Write to cache and DB together on every write | Data that must always be fresh in cache |
| **Write-behind** | Write to cache; async sync to DB | High-write throughput; risk of data loss |
| **Read-through** | Cache layer handles DB fetch transparently | Managed cache services (ElastiCache) |
| **Refresh-ahead** | Background job refreshes cache before expiry | Predictable access patterns |

**These notebooks simulate the DB** with a SQLite in-memory database (~50ms latency) to show real before/after cache performance comparisons.

---

In [ ]:
import fakeredis, json, time, hashlib

r = fakeredis.FakeRedis(decode_responses=True)

# Simulate a slow database
import sqlite3
db = sqlite3.connect(':memory:')
db.row_factory = sqlite3.Row
db.executescript("""
CREATE TABLE patients (
    patient_id TEXT PRIMARY KEY, name TEXT, age INT,
    province TEXT, last_hba1c REAL
);
INSERT INTO patients VALUES
    ('P001','Alice Ngata',   45,'NB',7.2),
    ('P002','Bob Chen',      52,'ON',6.8),
    ('P003','Fatima Rashid', 38,'BC',9.1),
    ('P004','James MacLeod', 61,'NS',5.6),
    ('P005','Mei-Ling Tran', 58,'QC',8.9);
""")
db.commit()

DB_QUERY_MS = 50  # simulated DB latency (ms)

def db_get_patient(patient_id):
    """Simulate a slow DB query."""
    time.sleep(DB_QUERY_MS / 1000)
    row = db.execute("SELECT * FROM patients WHERE patient_id=?", (patient_id,)).fetchone()
    return dict(row) if row else None

# ── Cache-aside (lazy loading) pattern ───────────────────────────
def get_patient(patient_id, ttl=300):
    cache_key = f"patient:{patient_id}"
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), 'HIT'
    # Cache miss: fetch from DB, store in cache
    data = db_get_patient(patient_id)
    if data:
        r.set(cache_key, json.dumps(data), ex=ttl)
    return data, 'MISS'

print("=== Cache-aside pattern ===")
print()
for pid in ['P001', 'P002', 'P001', 'P003', 'P001']:
    t0 = time.perf_counter()
    data, status = get_patient(pid)
    ms = (time.perf_counter() - t0) * 1000
    print(f"  get_patient({pid!r}): {status:<5s}  {ms:5.1f}ms  name={data['name']}")

print()
print("Cache-aside rules:")
rules = [
    ("Read path",    "Check cache first → miss → fetch DB → store in cache → return"),
    ("Write path",   "Write to DB → DELETE (or update) cache key (never write-to-cache-first)"),
    ("TTL",          "Set TTL on every cached key — prevents stale data living forever"),
    ("Cold start",   "First request always a miss; pre-warm cache for predictable latency"),
    ("Cache key",    "Use stable, deterministic keys: f'patient:{patient_id}'"),
    ("JSON",         "Serialise objects to JSON strings for string-type cache values"),
]
for label, rule in rules:
    print(f"  {label:<14s}  {rule}")


---
## TTL management and expiry patterns

In [ ]:
import time

print("=== TTL management and expiry patterns ===")
print()

# SET with TTL
r.set('session:tok123', json.dumps({'user':'P001','role':'patient'}), ex=1800)  # 30 min
r.set('otp:P001',       '847291',                                    ex=300)   # 5 min OTP
r.set('report:2024-Q4', json.dumps({'revenue': 142000}),             ex=86400) # 24h

print("TTL values:")
for key in ['session:tok123', 'otp:P001', 'report:2024-Q4']:
    ttl = r.ttl(key)
    pttl = r.pttl(key)
    print(f"  {key:<24s}  TTL={ttl}s  ({pttl}ms)")

# EXPIRE / PERSIST
r.set('temp_flag', 'true')
r.expire('temp_flag', 60)
print(f"\nAfter EXPIRE 60s: TTL={r.ttl('temp_flag')}s")
r.persist('temp_flag')
print(f"After PERSIST:     TTL={r.ttl('temp_flag')} (-1 = no expiry)")

# TTL return codes
r.set('no_ttl', 'value')
r.set('with_ttl', 'value', ex=10)
print()
print("TTL return codes:")
print(f"  Key with no TTL:      TTL={r.ttl('no_ttl')} (-1 = persistent)")
print(f"  Key with TTL:         TTL={r.ttl('with_ttl')}s")
print(f"  Key that doesn't exist: TTL={r.ttl('nonexistent')} (-2 = missing)")

# EXPIREAT — expire at a specific Unix timestamp
import datetime
tomorrow = int((datetime.datetime.now() + datetime.timedelta(days=1)).timestamp())
r.set('daily_summary', 'some_data')
r.expireat('daily_summary', tomorrow)
print(f"\nEXPIREAT tomorrow: TTL={r.ttl('daily_summary')}s (~{r.ttl('daily_summary')//3600}h)")

print()
print("TTL strategy guide:")
strategies = [
    ("Session tokens",      "30min–24h  ex=1800–86400"),
    ("OTP / verification",  "5min       ex=300"),
    ("API rate limit window","1min       ex=60"),
    ("Query result cache",  "5–60min    ex=300–3600"),
    ("Aggregated reports",  "1–24h      ex=3600–86400"),
    ("User preferences",    "7–30 days  ex=604800–2592000"),
    ("Leaderboard/ranking", "No TTL     PERSIST (updated in place)"),
    ("Pub/Sub messages",    "No TTL     (messages are transient by design)"),
]
print(f"  {'Use case':<26s}  Recommended TTL")
print("  " + "-"*50)
for use_case, ttl in strategies:
    print(f"  {use_case:<26s}  {ttl}")


---
## Cache stampede prevention

In [ ]:
import threading, time

print("=== Cache stampede prevention ===")
print()

print("The stampede problem:")
print("  1. A hot cache key expires")
print("  2. 100 concurrent requests all get a cache MISS simultaneously")
print("  3. All 100 hit the database at the same time -> overload")
print()

# Strategy A: SET NX lock (optimistic locking)
def get_with_lock(patient_id, ttl=300, lock_ttl=5):
    cache_key = f"patient:{patient_id}"
    lock_key  = f"lock:{cache_key}"
    cached = r.get(cache_key)
    if cached:
        return json.loads(cached), 'HIT'
    # Try to acquire lock (only one thread wins)
    acquired = r.set(lock_key, '1', nx=True, ex=lock_ttl)
    if acquired:
        try:
            data = db_get_patient(patient_id)
            if data:
                r.set(cache_key, json.dumps(data), ex=ttl)
            return data, 'MISS+FILL'
        finally:
            r.delete(lock_key)
    else:
        # Another thread is filling the cache — wait briefly and retry
        time.sleep(0.05)
        cached = r.get(cache_key)
        return (json.loads(cached), 'WAIT+HIT') if cached else (None, 'WAIT+MISS')

print("Strategy A: SET NX lock (one DB query wins the lock):")
results = []
threads = [threading.Thread(target=lambda: results.append(get_with_lock('P002')))
           for _ in range(5)]
# Sequential simulation since fakeredis is single-threaded here
for t in threads:
    t.start()
    t.join()
for data, status in results:
    print(f"  {status:<12s}  name={data['name'] if data else None}")

print()
# Strategy B: Probabilistic early expiration (PER)
print("Strategy B: Probabilistic Early Expiration (PER):")
print("  - Store (value, creation_time, TTL) together")
print("  - Before key expires, a random request 'thinks' it's expired")
print("  - That request refreshes the cache before the key actually expires")
print("  - Prevents simultaneous expiry storms without locking overhead")

per_example = """
import random, time, math

def fetch_with_per(key, fetch_fn, ttl=300, beta=1.0):
    cached = r.get(key + ':data')
    created_str = r.get(key + ':created')
    if cached and created_str:
        created = float(created_str)
        age     = time.time() - created
        # PER: randomly decide to refresh before actual expiry
        should_refresh = random.random() < math.exp(beta * (age - ttl) / ttl)
        if not should_refresh:
            return json.loads(cached), 'HIT'
    # Cache miss or PER-triggered refresh
    data = fetch_fn()
    now = str(time.time())
    r.set(key + ':data',    json.dumps(data), ex=ttl)
    r.set(key + ':created', now,              ex=ttl)
    return data, 'REFRESH'
"""
print(per_example)


---
## Common Pitfalls

**1. Writing to cache before writing to DB (write-through gone wrong)**
Writing to Redis first, then the DB means a DB write failure leaves stale data in the cache. Always write to the DB first, then update or DELETE the cache key. Deleting (invalidating) is safer than updating — the next read will re-populate correctly.

**2. Cache stampede on popular key expiry**
When a hot cache key expires, all concurrent readers miss simultaneously and hammer the DB. Use the SET NX lock pattern or probabilistic early expiration to prevent stampedes. Alternatively, pre-warm the cache before expiry with a background job.

**3. Caching NULL results (negative caching)**
A missing patient ID that returns `None` from the DB, not cached, leads to a DB hit on every request for that non-existent ID. Cache `None` as a sentinel: `r.set(key, '__null__', ex=60)` and check for the sentinel on cache hit.

**4. Using the same TTL for all data types**
A session token needs 30 minutes; a lab result summary can be cached for 1 hour; a patient name barely changes and can be cached for 24 hours. Tailor TTLs to data volatility. One-size-fits-all TTL causes over-caching of volatile data or under-caching of stable data.

**5. Not handling Redis downtime gracefully**
If Redis is unavailable, the application should fall through to the database — not crash. Wrap cache reads in try/except and degrade gracefully: slower responses are better than errors. Add circuit-breaker logic for sustained Redis unavailability.

**6. Cache invalidation on multi-field updates**
If a patient's name and age are cached together in one JSON key, updating only the age in the DB but setting just the age field in cache creates a split-brain cache. On any update, DELETE the entire cached object and let the next read re-fetch all fields.


---
*sql_methods_library - Samantha McGarrigle*